# 01 Data Understanding

## 1. Introduction

This notebook starts the Firefox Growth Intelligence project.

The goal of the project is to simulate how a product data scientist would help a browser team understand growth, product usage, experimentation, forecasting, and product-quality risk.

This first notebook focuses on one real Mozilla public dataset: the Mozilla SSL Ratios dataset.

We are not building a model yet. We are doing the responsible first step:

``` text
Understand the data
→ inspect quality
→ translate columns into product metrics
→ decide how this dataset can support the project

The key business question is:

Can this real Mozilla dataset support product trend analysis and forecasting?

## 2. Dataset Source

The dataset used in this notebook is the Mozilla SSL Ratios public dataset.

Dataset reference:

https://docs.telemetry.mozilla.org/datasets/other/ssl/reference.html

Public data access:

https://docs.telemetry.mozilla.org/cookbooks/public_data

Local file path:

```text
data/external/mozilla_ssl_ratios.json

This dataset tracks Firefox page-load activity split between:

SSL/HTTPS page loads
non-SSL/non-HTTPS page loads

That matters because SSL/HTTPS usage is connected to browser trust, web security, and product health.

For this project, this dataset will be used for:

real Firefox trend analysis
forecasting preparation
dashboard metrics
product-health storytelling

Important limitation:

This dataset does not contain user-level experiment assignments, onboarding behavior, or retention outcomes.

So later:

Real Mozilla SSL data
→ trend analysis and forecasting

Synthetic user-level data
→ A/B testing, causal inference, retention modeling

That keeps the project honest and professional.

## 3. Load Dataset

In this section, we load the Mozilla SSL Ratios JSON file into pandas.

Before doing any analysis, we need to confirm:

- whether the file loads correctly
- how many rows and columns it contains
- what the first few records look like
- whether the dataset is line-delimited JSON or standard JSON

This step is important because real public datasets are not always shaped like clean CSV files.

In [ ]:
# Import core libraries for data analysis
from pathlib import Path

import pandas as pd
import numpy as np

In [ ]:
# Define the dataset path
DATA_PATH = Path("../data/external/mozilla_ssl_ratios.json")

# Check that the file exists before loading
DATA_PATH.exists()

In [ ]:
# Load the Mozilla SSL Ratios dataset
# The file is line-delimited JSON, so we use lines=True
df = pd.read_json(DATA_PATH)

# Display the first few rows
df.head()

In [ ]:
# Display the last 5 rows
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

Basic Inspection

In [ ]:
# Check column names and data types
df.info()

In [ ]:
# Count missing values by column
df.isna().sum()

In [ ]:
# Check duplicate rows
df.duplicated().sum()

In [ ]:
# Basic summary statistics for numeric columns
df.describe()

Column Meaning

The raw dataset has these columns:

submission_date
os
country
non_ssl_loads
ssl_loads
reporting_ratio

Product meaning:

submission_date
→ date the data was aggregated

os
→ operating system segment

country
→ country segment

non_ssl_loads
→ page loads not using SSL/HTTPS

ssl_loads
→ page loads using SSL/HTTPS

reporting_ratio
→ adjustment factor connected to telemetry reporting population

The raw data is useful, but it is not yet in the best shape for product analytics.
We need to engineer product metrics:

total_page_loads
→ ssl_loads + non_ssl_loads

ssl_ratio
→ ssl_loads / total_page_loads

normalized_page_loads
→ total_page_loads / reporting_ratio

These engineered metrics give us:

security/trust trend
usage activity proxy
forecasting-ready time series

### Missing Value Result After Cleaning

After removing rows where `ssl_loads` or `non_ssl_loads` were missing, the main product metric columns are complete.

Remaining missing values:

- `country`: 6,113 missing values

This is acceptable for overall trend analysis because country is only needed for country-level segmentation. For overall SSL ratio trends and forecasting, these rows can remain.

For country-level analysis later, rows with missing `country` values will be filtered out.

In [ ]:
# Create product metric columns from the cleaned dataset

# Total page loads combine SSL and non-SSL page loads
df_clean["total_page_loads"] = df_clean["ssl_loads"] + df_clean["non_ssl_loads"]

# SSL ratio measures the share of page loads conducted over SSL/HTTPS
df_clean["ssl_ratio"] = df_clean["ssl_loads"] / df_clean["total_page_loads"]

# Normalized page loads adjust total page-load activity by reporting ratio
df_clean["normalized_page_loads"] = df_clean["total_page_loads"] / df_clean["reporting_ratio"]

# Display the updated dataset
df_clean.head()

In [ ]:
# Confirm the engineered columns were created correctly
df_clean[
    [
        "non_ssl_loads",
        "ssl_loads",
        "total_page_loads",
        "ssl_ratio",
        "normalized_page_loads",
    ]
].describe()

### Missing Value Interpretation

The dataset has no missing values in `submission_date`, `os`, or `reporting_ratio`.

The main missingness appears in `non_ssl_loads` and `ssl_loads`. Since these two fields are required to calculate total page loads and SSL ratio, rows missing either value should be removed before trend analysis and forecasting.

The `country` column has a smaller number of missing values. These rows can remain for overall trend analysis, but should be filtered out when performing country-level segment analysis.

We will not replace missing page-load values with zero because doing so could distort SSL ratio and page-load trend calculations.

## 8. Product Metric Trends

This section explores the main product metrics over time.

The engineered metrics are:

- `ssl_ratio`: share of page loads using SSL/HTTPS
- `normalized_page_loads`: page-load activity adjusted by reporting ratio
- `total_page_loads`: raw SSL + non-SSL page-load activity

For this project, `ssl_ratio` represents a browser security and trust trend, while `normalized_page_loads` acts as a public usage activity proxy.